Cell 1 — Saare cities combine karo

In [1]:
import pandas as pd
import numpy as np

cities = ['Bangalore', 'Chennai', 'Delhi', 'Hyderabad', 'Kolkata', 'Mumbai']

dfs = []
for city in cities:
    df = pd.read_csv(f'../data/raw/{city}.csv')
    df['city'] = city  # city column add karo
    dfs.append(df)

master = pd.concat(dfs, ignore_index=True)
print("Combined shape:", master.shape)
print("\nCity counts:")
print(master['city'].value_counts())
print("\nColumns:", master.columns.tolist())
print("\nMissing values:")
print(master.isnull().sum()[master.isnull().sum() > 0])

Combined shape: (32963, 41)

City counts:
city
Mumbai       7719
Kolkata      6507
Bangalore    6207
Chennai      5014
Delhi        4998
Hyderabad    2518
Name: count, dtype: int64

Columns: ['Price', 'Area', 'Location', 'No. of Bedrooms', 'Resale', 'MaintenanceStaff', 'Gymnasium', 'SwimmingPool', 'LandscapedGardens', 'JoggingTrack', 'RainWaterHarvesting', 'IndoorGames', 'ShoppingMall', 'Intercom', 'SportsFacility', 'ATM', 'ClubHouse', 'School', '24X7Security', 'PowerBackup', 'CarParking', 'StaffQuarter', 'Cafeteria', 'MultipurposeRoom', 'Hospital', 'WashingMachine', 'Gasconnection', 'AC', 'Wifi', "Children'splayarea", 'LiftAvailable', 'BED', 'VaastuCompliant', 'Microwave', 'GolfCourse', 'TV', 'DiningTable', 'Sofa', 'Wardrobe', 'Refrigerator', 'city']

Missing values:
Series([], dtype: int64)


Cell 2 — Clean karo

In [2]:
# Sirf useful columns rakho
cols = ['Price', 'Area', 'Location', 'No. of Bedrooms', 'city',
        'Gymnasium', 'SwimmingPool', 'CarParking', 'LiftAvailable',
        'PowerBackup', '24X7Security', 'Resale']

df = master[cols].copy()

# Rename for simplicity
df.rename(columns={
    'No. of Bedrooms': 'bhk',
    'Area': 'total_sqft',
    'Price': 'price',
    'Location': 'location'
}, inplace=True)

# Missing values hatao
df.dropna(inplace=True)

# Price 0 ya negative nahi hona chahiye
df = df[df['price'] > 0]
df = df[df['total_sqft'] > 0]
df = df[df['bhk'] > 0]

# Price ko lakhs mein convert karo
# Pehle dekho price ka range
print("Price stats:")
print(df['price'].describe())
print("\nSample prices:", df['price'].head(10).tolist())

Price stats:
count    3.296300e+04
mean     1.168672e+07
std      2.307368e+07
min      2.000000e+06
25%      4.071500e+06
50%      6.711000e+06
75%      1.200000e+07
max      8.546000e+08
Name: price, dtype: float64

Sample prices: [30000000, 7888000, 4866000, 8358000, 6845000, 6797000, 20000000, 7105000, 8405000, 3506000]


Cell 3 — Price unit fix karo

In [3]:
# Price range dekh ke decide karo
max_price = df['price'].max()
print(f"Max price: {max_price}")
print(f"Min price: {df['price'].min()}")
print(f"Mean price: {df['price'].mean():.0f}")

# Agar price crores mein hai (>1,00,00,000) toh lakhs mein convert karo
if max_price > 1_00_00_000:
    df['price'] = df['price'] / 1_00_000  # rupees → lakhs
    print("\nConverted to Lakhs")
elif max_price > 1_000:
    df['price'] = df['price'] / 100  # already thousands
    print("\nDivided by 100")

print(f"\nAfter conversion - Mean price: {df['price'].mean():.2f} Lakhs")
print(df[['city', 'price', 'total_sqft', 'bhk']].head(5))

Max price: 854599999
Min price: 2000000
Mean price: 11686718

Converted to Lakhs

After conversion - Mean price: 116.87 Lakhs
        city   price  total_sqft  bhk
0  Bangalore  300.00        3340    4
1  Bangalore   78.88        1045    2
2  Bangalore   48.66        1179    2
3  Bangalore   83.58        1675    3
4  Bangalore   68.45        1670    3


Cell 4 — Outliers hatao

In [4]:
# Per sqft calculate karo
df['price_per_sqft'] = (df['price'] * 100000) / df['total_sqft']

print("Price per sqft stats:")
print(df['price_per_sqft'].describe())

# Citywise outliers remove karo
def remove_outliers(df):
    result = []
    for city in df['city'].unique():
        city_df = df[df['city'] == city].copy()
        m  = city_df['price_per_sqft'].mean()
        sd = city_df['price_per_sqft'].std()
        cleaned = city_df[
            (city_df['price_per_sqft'] > m - 2*sd) &
            (city_df['price_per_sqft'] < m + 2*sd)
        ]
        result.append(cleaned)
        print(f"{city}: {len(city_df)} → {len(cleaned)} rows")
    return pd.concat(result, ignore_index=True)

df = remove_outliers(df)
print(f"\nFinal shape: {df.shape}")

Price per sqft stats:
count    3.296300e+04
mean     9.746699e+03
std      1.823321e+04
min      2.410000e+02
25%      4.000000e+03
50%      5.775556e+03
75%      9.500000e+03
max      1.076444e+06
Name: price_per_sqft, dtype: float64
Bangalore: 6207 → 6119 rows
Chennai: 5014 → 4885 rows
Delhi: 4998 → 4917 rows
Hyderabad: 2518 → 2439 rows
Kolkata: 6507 → 6351 rows
Mumbai: 7719 → 7445 rows

Final shape: (32156, 13)


Cell 5 — Encoding

In [5]:
from sklearn.preprocessing import LabelEncoder
import joblib, os

os.makedirs('../data/processed', exist_ok=True)

# City encoder
city_enc = LabelEncoder()
df['city_enc'] = city_enc.fit_transform(df['city'])
joblib.dump(city_enc, '../data/processed/city_encoder.pkl')

# Location encoder
loc_enc = LabelEncoder()
df['location_enc'] = loc_enc.fit_transform(df['location'])
joblib.dump(loc_enc, '../data/processed/location_encoder_multi.pkl')

print("Cities:", city_enc.classes_.tolist())
print("Total locations:", len(loc_enc.classes_))

# Final features
feature_cols = ['city_enc', 'location_enc', 'total_sqft', 'bhk',
                'price_per_sqft', 'Gymnasium', 'SwimmingPool',
                'CarParking', 'LiftAvailable', 'PowerBackup',
                '24X7Security', 'Resale']

df_model = df[feature_cols + ['price']].copy()
df_model.to_csv('../data/processed/clean_data_multi.csv', index=False)
print("\nSaved! Shape:", df_model.shape)
print(df_model.head(3))

Cities: ['Bangalore', 'Chennai', 'Delhi', 'Hyderabad', 'Kolkata', 'Mumbai']
Total locations: 1736

Saved! Shape: (32156, 13)
   city_enc  location_enc  total_sqft  bhk  price_per_sqft  Gymnasium  \
0         0           578        3340    4     8982.035928          1   
1         0           364        1045    2     7548.325359          1   
2         0           678        1179    2     4127.226463          1   

   SwimmingPool  CarParking  LiftAvailable  PowerBackup  24X7Security  Resale  \
0             1           0              1            1             1       0   
1             1           1              1            1             1       0   
2             1           0              1            0             1       0   

    price  
0  300.00  
1   78.88  
2   48.66  



Cell 6 — Multi-city model train karo

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
import xgboost as xgb

X = df_model.drop('price', axis=1)
y = np.log1p(df_model['price'])  # log transform

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=100)

# Accuracy
y_pred = np.expm1(model.predict(X_test))
y_real = np.expm1(y_test)

mae = mean_absolute_error(y_real, y_pred)
r2  = r2_score(y_real, y_pred)

print(f"\nMAE      : {mae:.2f} Lakhs")
print(f"R2 Score : {r2:.4f}")
print(f"Accuracy : {round(r2*100, 2)}%")

[0]	validation_0-rmse:0.69598
[100]	validation_0-rmse:0.05674
[200]	validation_0-rmse:0.04148
[300]	validation_0-rmse:0.03936
[400]	validation_0-rmse:0.03843
[499]	validation_0-rmse:0.03788

MAE      : 3.36 Lakhs
R2 Score : 0.9311
Accuracy : 93.11%


Cell 7 — Save karo

In [7]:
import json

joblib.dump(model, '../data/processed/model_multi.pkl')

meta = {
    "features"         : feature_cols,
    "use_log_transform": True,
    "price_unit"       : "lakhs",
    "cities"           : city_enc.classes_.tolist(),
    "inflation_factor" : 1.35,
    "version"          : "2.0_multicity"
}

with open('../data/processed/model_meta_multi.json', 'w') as f:
    json.dump(meta, f, indent=2)

print("Saved:")
print("  model_multi.pkl")
print("  model_meta_multi.json")
print("  city_encoder.pkl")
print("  location_encoder_multi.pkl")

# Quick test
sample = pd.DataFrame([{
    'city_enc'      : city_enc.transform(['Mumbai'])[0],
    'location_enc'  : 0,
    'total_sqft'    : 1200,
    'bhk'           : 2,
    'price_per_sqft': 8000,
    'Gymnasium'     : 1,
    'SwimmingPool'  : 0,
    'CarParking'    : 1,
    'LiftAvailable' : 1,
    'PowerBackup'   : 1,
    '24X7Security'  : 1,
    'Resale'        : 0,
}])

pred = np.expm1(model.predict(sample)[0]) * 1.35
print(f"\nMumbai 2BHK 1200sqft → ₹{round(pred, 2)} Lakhs")

Saved:
  model_multi.pkl
  model_meta_multi.json
  city_encoder.pkl
  location_encoder_multi.pkl

Mumbai 2BHK 1200sqft → ₹143.23 Lakhs
